In [ ]:
%gui qt
%load_ext autoreload
%autoreload 2

In [ ]:
import hmt_v3 as hmt
import numpy as np
import pandas as pd

In [ ]:
me3_raw_df = pd.read_csv("test_data/k27_k27_thaw009_me3.csv")
ac_raw_df = pd.read_csv("test_data/k27_k27_thaw009_ac.csv")

me3_filtered_df = hmt.preprocess.filter_axial(me3_raw_df)
ac_filtered_df = hmt.preprocess.filter_axial(ac_raw_df)

binary_mask, me3_df, ac_df = hmt.preprocess.binarize_nucleus(me3_filtered_df, ac_filtered_df, thresh=2, bin_size=50, show_plots=True)
distance_map, contour_bands = hmt.preprocess.create_radial_contours(binary_mask, show_plots=True)

In [ ]:
step=10

print("Extracting emperical H3K27me3 distributions...")
me3_rdf, me3_adf = hmt.simulate.extract_empirical_parameters(me3_df, sdis=500, step=step)

print("Extracting emperical H3K27ac distributions...")
ac_rdf, ac_adf = hmt.simulate.extract_empirical_parameters(ac_df, sdis=500, step=step)

In [ ]:
hmt.visualize.plot_rdf_adf(me3_rdf, me3_adf, ac_rdf, ac_adf, step=10)

In [ ]:
me3_n = hmt.simulate.extract_n_locs_from_rdf(me3_rdf, step=step)
ac_n  = hmt.simulate.extract_n_locs_from_rdf(ac_rdf,  step=step)

print(f"Mean Number of Localizations per nanodomain: \nH3K27me3: {me3_n} \nH3K27ac: {ac_n}")

me3_seeds = [[0, 0, 0]]
me3_locs = hmt.simulate.spawn_nanodomains(me3_seeds, n_locs=me3_n, step=step, rdf=me3_rdf, adf=me3_rdf)

ac_seeds = [[900, 0, 0]]
ac_locs = hmt.simulate.spawn_nanodomains(ac_seeds, n_locs=ac_n, step=step, rdf=ac_rdf, adf=ac_rdf, start_label=len(me3_seeds))

seeds = np.concat([me3_seeds, ac_seeds])
locs = pd.concat([me3_locs, ac_locs], ignore_index=True)
hmt.visualize.plot_nanodomain_2d(locs, seeds)
hmt.visualize.plot_nanodomain_3d(locs, seeds)

In [ ]:
me3_df = hmt.simulate.assign_contour_bands(me3_df, contour_bands, bin_size=50)
ac_df  = hmt.simulate.assign_contour_bands(ac_df,  contour_bands, bin_size=50)

inner_me3 = me3_df[me3_df['contour_band'] <= 20]
outer_me3 = me3_df[me3_df['contour_band'] > 80 ]

inner_ac  = ac_df[ac_df['contour_band'] <= 20]
outer_ac  = ac_df[ac_df['contour_band'] > 80 ]

print("Extracting emperical H3K27me3 distributions...")
inner_me3_rdf, inner_me3_adf = hmt.simulate.extract_empirical_parameters(inner_me3, sdis=500, step=step)
outer_me3_rdf, outer_me3_adf = hmt.simulate.extract_empirical_parameters(outer_me3, sdis=500, step=step)

print("Extracting emperical H3K27ac distributions...")
inner_ac_rdf,  inner_ac_adf  = hmt.simulate.extract_empirical_parameters(inner_ac, sdis=500, step=step)
outer_ac_rdf,  outer_ac_adf  = hmt.simulate.extract_empirical_parameters(outer_ac, sdis=500, step=step)

In [ ]:
hmt.visualize.plot_rdf_adf(inner_me3_rdf, inner_me3_adf, inner_ac_rdf, inner_ac_adf)
hmt.visualize.plot_rdf_adf(outer_me3_rdf, outer_me3_adf, outer_ac_rdf, outer_ac_adf)

inner_me3_n = hmt.simulate.extract_n_locs_from_rdf(inner_me3_rdf, step=step)
inner_ac_n  = hmt.simulate.extract_n_locs_from_rdf(inner_ac_rdf,  step=step)
print(f"Mean Number of Localizations per nanodomain from inner rings: \nH3K27me3: {inner_me3_n} \nH3K27ac: {inner_ac_n}\n")

outer_me3_n = hmt.simulate.extract_n_locs_from_rdf(outer_me3_rdf, step=step)
outer_ac_n  = hmt.simulate.extract_n_locs_from_rdf(outer_ac_rdf,  step=step)
print(f"Mean Number of Localizations per nanodomain from outer rings: \nH3K27me3: {outer_me3_n} \nH3K27ac: {outer_ac_n}")

In [ ]:
# 1. Preprocess
binary_mask, me3_masked, ac_masked = hmt.preprocess.binarize_nucleus(me3_df, ac_df, thresh=2)
distance_map, contour_bands = hmt.preprocess.create_radial_contours(binary_mask)
x_min = pd.concat([me3_df, ac_df])["x [nm]"].min()
y_min = pd.concat([me3_df, ac_df])["y [nm]"].min()

# 2. Extract radial profile from the channel you want to simulate
profile = hmt.simulate.extract_radial_density_profile(me3_masked, contour_bands, x_min, y_min)

# 3. Place seeds
seeds = hmt.simulate.place_seeds(contour_bands, profile, me3_masked["z [nm]"].values, x_min, y_min,
                    scaling_factor=0.01)  # tune this to match domain count

# 4. Spawn localizations around each seed
rdf, adf = hmt.simulate.extract_empirical_parameters(me3_masked)
n_locs = hmt.simulate.extract_n_locs_from_rdf(rdf)
sim_df = hmt.simulate.spawn_nanodomains(seeds, rdf=rdf, adf=adf, n_locs=n_locs)

hmt.visualize.plot_nanodomain_2d(sim_df, seeds, title='H3K27me3 Nanodomains',plot_seeds=False)

In [ ]:
# 2. Extract radial profile from the channel you want to simulate
profile = hmt.simulate.extract_radial_density_profile(ac_masked, contour_bands, x_min, y_min)

# 3. Place seeds
seeds = hmt.simulate.place_seeds(contour_bands, profile, ac_masked["z [nm]"].values, x_min, y_min,
                    scaling_factor=0.01)  # tune this to match domain count

# 4. Spawn localizations around each seed
rdf, adf = hmt.simulate.extract_empirical_parameters(ac_masked)
n_locs = hmt.simulate.extract_n_locs_from_rdf(rdf)
sim_df = hmt.simulate.spawn_nanodomains(seeds, rdf=rdf, adf=adf, n_locs=n_locs)

hmt.visualize.plot_nanodomain_2d(sim_df, seeds, title='H3K27ac Nanodomains',plot_seeds=False)

In [ ]:
step = 10

# ── 1. Preprocess ─────────────────────────────────────────────────────────────
x_min = pd.concat([me3_df, ac_df])["x [nm]"].min()
y_min = pd.concat([me3_df, ac_df])["y [nm]"].min()

# ── 2. Extract empirical parameters ───────────────────────────────────────────
me3_rdf, me3_adf = hmt.simulate.extract_empirical_parameters(me3_df, sdis=500, step=step)
ac_rdf,  ac_adf  = hmt.simulate.extract_empirical_parameters(ac_df,  sdis=500, step=step)

me3_profile = hmt.simulate.extract_radial_density_profile(me3_df, contour_bands, x_min, y_min)
ac_profile  = hmt.simulate.extract_radial_density_profile(ac_df,  contour_bands, x_min, y_min)

# ── 3. Estimate noise fractions ────────────────────────────────────────────────
print("H3K27me3:")
me3_f_noise = hmt.simulate.estimate_noise_fraction(me3_df, contour_bands, x_min, y_min,
                                                    rdf=me3_rdf, step=step, verbose=True)
print("H3K27ac:")
ac_f_noise  = hmt.simulate.estimate_noise_fraction(ac_df,  contour_bands, x_min, y_min,
                                                    rdf=ac_rdf,  step=step, verbose=True)

me3_f_domain = 1 - me3_f_noise
ac_f_domain  = 1 - ac_f_noise

print(f"\nNoise fraction — me3: {me3_f_noise:.2f}, ac: {ac_f_noise:.2f}")

# ── 4. Simulate ────────────────────────────────────────────────────────────────
me3_n_locs_rdf = hmt.simulate.extract_n_locs_from_rdf(me3_rdf, step=step)
ac_n_locs_rdf  = hmt.simulate.extract_n_locs_from_rdf(ac_rdf,  step=step)

me3_n_locs_true = me3_n_locs_rdf / me3_f_domain
ac_n_locs_true  = ac_n_locs_rdf  / ac_f_domain

me3_scaling = me3_f_domain**2 / me3_n_locs_rdf
ac_scaling  = ac_f_domain**2  / ac_n_locs_rdf

me3_seeds = hmt.simulate.place_seeds(contour_bands, me3_profile, me3_df["z [nm]"].values,
                                     x_min, y_min, scaling_factor=me3_scaling)
ac_seeds  = hmt.simulate.place_seeds(contour_bands, ac_profile,  ac_df["z [nm]"].values,
                                     x_min, y_min, scaling_factor=ac_scaling)

me3_sim = hmt.simulate.spawn_nanodomains(me3_seeds, rdf=me3_rdf, adf=me3_adf,
                                         n_locs=me3_n_locs_true, step=step)
ac_sim  = hmt.simulate.spawn_nanodomains(ac_seeds,  rdf=ac_rdf,  adf=ac_adf,
                                         n_locs=ac_n_locs_true,  step=step,
                                         start_label=len(me3_seeds))

me3_noise = hmt.simulate.add_noise_locs(int(me3_f_noise * len(me3_df)), contour_bands,
                                        me3_df["z [nm]"].values, x_min, y_min)
ac_noise  = hmt.simulate.add_noise_locs(int(ac_f_noise  * len(ac_df)),  contour_bands,
                                        ac_df["z [nm]"].values,  x_min, y_min)

me3_final = pd.concat([me3_sim, me3_noise], ignore_index=True)
ac_final  = pd.concat([ac_sim,  ac_noise],  ignore_index=True)

print(f"\nme3 — real: {len(me3_df):,}, simulated: {len(me3_final):,} ({len(me3_seeds):,} seeds)")
print(f"ac  — real: {len(ac_df):,}, simulated: {len(ac_final):,} ({len(ac_seeds):,} seeds)")

In [ ]:
hmt.visualize.plot_nanodomain_2d(me3_sim, me3_seeds, title='H3K27me3 Nanodomains',plot_seeds=False)
hmt.visualize.plot_nanodomain_2d(ac_sim, ac_seeds, title='H3K27ac Nanodomains',plot_seeds=False)


In [ ]:
hmt.visualize.plot_nanodomain_3d(me3_sim, me3_seeds, title='H3K27me3 Nanodomains',plot_seeds=False)
hmt.visualize.plot_nanodomain_3d(ac_sim, ac_seeds, title='H3K27ac Nanodomains',plot_seeds=False)
